In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

In [2]:
FOOT_MARKERS = [
    "CAL1",
    "CUB",
    "LCAL",
    "LMAL",
    "MCAL",
    "MMAL",
    "MT1B",
    "MT1H",
    "MT2H",
    "MT5B",
    "MT5H",
    "NAV",
    "TOE",
]
FOOT_MARKERS = [p + x for x in FOOT_MARKERS for p in ("L", "R")]

filenames = [
    "inc0_10kmh.tsv",
    "inc0_12kmh.tsv",
    "inc0_14kmh.tsv",
    "inc5_10kmh.tsv",
    "inc5_12kmh.tsv",
    "inc5_14kmh.tsv",
    "inc10_10kmh.tsv",
    "inc10_12kmh.tsv",
    "inc10_14kmh.tsv",
]

for filename in filenames:
    print(f"Processing {filename}...")

    # ignore first 10 lines of metadata
    df = pd.read_csv(f"Incline Running/{filename}", sep="\t", skiprows=10)
    df = df.iloc[:, :-1]  # drops last column which is empty
    df = df.iloc[:1000, :]  # only keep first 5 seconds of data
    # df = df.iloc[::10, :]  # downsample to 20Hz

    df_x = df[df.columns[df.columns.str.endswith("X")]].rename(columns=lambda x: x[:-2])
    df_y = df[df.columns[df.columns.str.endswith("Y")]].rename(columns=lambda x: x[:-2])
    df_z = df[df.columns[df.columns.str.endswith("Z")]].rename(columns=lambda x: x[:-2])

    # convert to long format
    df_x_long = df_x.melt(var_name="Marker", value_name="X", ignore_index=False)
    df_y_long = df_y.melt(var_name="Marker", value_name="Y", ignore_index=False)
    df_z_long = df_z.melt(var_name="Marker", value_name="Z", ignore_index=False)

    # rename index to Time
    df_x_long.index.name = "Time"
    df_y_long.index.name = "Time"
    df_z_long.index.name = "Time"

    # merge x, y, z dataframes on index and marker name
    df_long = df_x_long.merge(df_y_long, on=["Time", "Marker"]).merge(
        df_z_long, on=["Time", "Marker"]
    )
    df_long.reset_index(inplace=True)

    # add color column based on marker name
    colors = {m: "#888888" for m in df_long["Marker"].unique()}
    colors.update({m: "#1F1F1F" for m in FOOT_MARKERS})
    colors.update({m: "#46d1de" for m in ["RCAL1", "LCAL1"]})
    colors.update({m: "#cfad3f" for m in ["RCUB", "LCUB"]})
    colors.update({m: "#27c15d" for m in ["RMT2H", "LMT2H"]})
    colors.update({m: "#ce4a4a" for m in ["RLCAL", "LLCAL"]})
    colors.update({m: "#5037b4" for m in ["RTOE", "LTOE"]})
    df_long["Color"] = df_long["Marker"].map(colors)

    # create 3D scatter plot with animation
    fig = px.scatter_3d(
        df_long,
        x="X",
        y="Y",
        z="Z",
        color="Marker",
        animation_frame="Time",
        color_discrete_map=colors,
    )
    fig.write_html(f"figures/fancy_{filename}.html")

Processing inc0_10kmh.tsv...
Processing inc0_12kmh.tsv...
Processing inc0_14kmh.tsv...
Processing inc5_10kmh.tsv...
Processing inc5_12kmh.tsv...
Processing inc5_14kmh.tsv...
Processing inc10_10kmh.tsv...
Processing inc10_12kmh.tsv...
Processing inc10_14kmh.tsv...
